# SwingSight Exact Club-Marking CNN Training

This notebook trains the second-stage model that reads an iron number, wedge letter, or wedge loft after the five-way classifier identifies an Iron or Wedge.

It exports `models/trained/club_marking_cnn.pt`, the exact location SwingSight checks automatically.


In [ ]:
import sys
import subprocess

packages = ["torch>=2.2.0", "torchvision>=0.17.0", "pillow>=10.0.0", "numpy>=1.24.0", "matplotlib>=3.8.0"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "--no-warn-script-location"])
print("Training packages are ready.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SwingSight-AI project.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data" / "club_cnn" / "club_marking"
MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "club_marking_cnn.pt"
EXPERIMENT_DIR = PROJECT_ROOT / "outputs" / "experiments" / "club_marking"

print("Project:", PROJECT_ROOT)
print("Marking dataset:", DATA_DIR)
print("App model output:", MODEL_PATH)


## Dataset layout

Use tight, readable crops of the club's sole or face. Do **not** include a full swing frame for this model.

```text
data/club_cnn/club_marking/
  train/
    1/ ... 9/
    p/ a/ g/ s/ l/
    50/ 52/ 54/ 56/ 58/ 60/
  val/
    1/ ... 9/
    p/ a/ g/ s/ l/
    50/ 52/ 54/ 56/ 58/ 60/
```

The folder names are the model labels. Use images from several club brands, lighting conditions, orientations, and backgrounds. Do not horizontally flip digits or letters—mirroring changes the marking.


In [ ]:
from collections import Counter

CLASS_NAMES = [
    "1", "2", "3", "4", "5", "6", "7", "8", "9",
    "p", "a", "g", "s", "l",
    "50", "52", "54", "56", "58", "60",
]
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def inventory(split: str) -> Counter:
    split_dir = DATA_DIR / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Missing {split_dir}")
    actual_classes = sorted(path.name for path in split_dir.iterdir() if path.is_dir())
    if set(actual_classes) != set(CLASS_NAMES):
        missing = sorted(set(CLASS_NAMES) - set(actual_classes))
        extra = sorted(set(actual_classes) - set(CLASS_NAMES))
        raise ValueError(f"{split} folders do not match. Missing={missing}; extra={extra}")
    return Counter({
        label: sum(1 for path in (split_dir / label).rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)
        for label in CLASS_NAMES
    })

train_counts = inventory("train")
val_counts = inventory("val")
print("Train images:", sum(train_counts.values()))
print("Validation images:", sum(val_counts.values()))
print("Smallest train classes:", train_counts.most_common()[-5:])


In [ ]:
if min(train_counts.values()) < 20 or min(val_counts.values()) < 5:
    raise ValueError(
        "Each class needs at least 20 training and 5 validation images before a useful first run. "
        "More variety is better, especially for common loft markings."
    )
print("Dataset passes the minimum size check.")


## Train

This calls the repository's training script so the checkpoint contains the exact format, class ordering, and preprocessing that `src/swingsight/club_recognition.py` validates at runtime.


In [ ]:
import subprocess

EPOCHS = 35
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

command = [
    sys.executable,
    "scripts/train_club_cnn.py",
    "--task", "club_marking",
    "--data-dir", str(DATA_DIR),
    "--output", str(MODEL_PATH),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--workers", "0",
]
subprocess.check_call(command, cwd=PROJECT_ROOT)
print("Saved:", MODEL_PATH)


In [ ]:
import torch

checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
print("Task:", checkpoint["task"])
print("Classes:", checkpoint["class_names"])
print("Validation accuracy:", checkpoint["validation_accuracy"])
assert checkpoint["task"] == "club_marking"
assert checkpoint["class_names"] == CLASS_NAMES


In [ ]:
from PIL import Image
from swingsight.club_cnn import classify_image

sample_images = [
    path
    for label in CLASS_NAMES
    for path in sorted((DATA_DIR / "val" / label).glob("*"))[:1]
    if path.suffix.lower() in IMAGE_SUFFIXES
]

for sample_path in sample_images[:12]:
    with Image.open(sample_path) as opened:
        prediction = classify_image(
            opened.convert("RGB"),
            checkpoint_path=MODEL_PATH,
            expected_task="club_marking",
            minimum_confidence=0.0,
        )
    print(f"actual={sample_path.parent.name:>2}  predicted={prediction.label:>2}  confidence={prediction.confidence:.3f}")


## App check

Restart SwingSight after training. For an Iron or Wedge, use a close, well-lit image of the sole/face with the number, letter, or loft centered. The result should now return a player-facing club such as **7 Iron**, **Pitching Wedge**, or **56° Wedge** instead of reporting a missing marking checkpoint.
